<a href="https://colab.research.google.com/github/halimajaved592-arch/urdu-ocr-codesaviours-si26-halima/blob/main/SI26_Week3_Halima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch pillow pandas


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd

In [ ]:
import os
import torch
import pandas as pd
from torch.utils.data import Dataset
from PIL import Image

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Full image path
        image_path = os.path.join(
            "/content/drive/MyDrive/Colab Notebooks",
            row["image"]
        )

        # Load image
        image = Image.open(image_path).convert("RGB")

        # Process image
        encoding = self.processor(image, return_tensors="pt")
        pixel_values = encoding.pixel_values.squeeze()

        # Process text
        labels = self.processor.tokenizer(
            row["text"],
            padding="max_length",
            max_length=128
        ).input_ids

        labels = torch.tensor(labels)

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import TrOCRProcessor
import torch

# Load processor
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")

# Create dataset
dataset = UrduOCRDataset(
    "/content/drive/MyDrive/Colab Notebooks/data/labels.csv",
    processor
)

# Test first sample
sample = dataset[0]

print("Sample pixel_values shape:", sample["pixel_values"].shape)
print("Sample labels shape:", sample["labels"].shape)
print("Dataset is working correctly!")

# Train/test split
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, test_size]
)

print("Training samples:", train_size)
print("Testing samples:", test_size)

Dataset loaded: 200 samples
Sample pixel_values shape: torch.Size([3, 384, 384])
Sample labels shape: torch.Size([128])
Dataset is working correctly!
Training samples: 160
Testing samples: 40
